### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="micro_mass",
    dataset_year="2013",
    domain_str="biology & life sciences",
    # Data Source
    dataset_source="UCI",
    original_dataset_source_download_link="https://doi.org/10.24432/C5T61S",
    download_description="""
wget https://archive.ics.uci.edu/static/public/253/micromass.zip && unzip micromass.zip micromass_un_anonymized.zip && rm micromass.zip && mkdir -p local-data-warehouse/micro_mass && unzip micromass_un_anonymized.zip -d local-data-warehouse/micro_mass/ && rm micromass_un_anonymized.zip
""",
    # References
    academic_reference_bibtex=r"""@article{mahe2014automatic,
  title={Automatic identification of mixed bacterial species fingerprints in a MALDI-TOF mass-spectrum},
  author={Mahe, Pierre and Arsac, Maud and Chatellier, Sonia and Monnin, Val{\'e}rie and Perrot, Nadine and Mailler, Sandrine and Girard, Victoria and Ramjeet, Mahendrasingh and Surre, J{\'e}r{\'e}my and Lacroix, Bruno and others},
  journal={Bioinformatics},
  volume={30},
  number={9},
  pages={1280--1286},
  year={2014},
  publisher={Oxford University Press}
}
""",
    academic_reference_bibtex_key="mahe2014automatic",
    license="CC BY 4.0",
    data_tags=["Non-IID", "Grouped"],
    curation_comments="""
We use the pure spectra version (without mixing) and replicate a grouped-based task to predict the species of strains.

- We drop columns with the same value across all samples.
- We drop constant columns (all 0).
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="Species",
    problem_type="multiclass_classification",
    objective_metric_name="",
    stratify_on="Species",
    group_on="Strain",
    group_labels="per_group",
)

## Preprocessing

In [2]:
import pandas as pd

df = pd.read_csv(dataset_mold.path / "micromass" / "pure_spectra_matrix.csv", header=None, names=[f"peak_list_spectra_bin_{i}" for i in range(1, 1301)], sep=";")
metadata = pd.read_csv(dataset_mold.path / "micromass" / "pure_spectra_metadata_with-species-name.csv", sep=";")
df = pd.concat([metadata, df], axis=1)
print("Loaded data shape:", df.shape)


as_cat_type = ["Strain", "Species"]
df[as_cat_type] = df[as_cat_type].astype("category")

# Drop duplicate columns based on values
df = df.loc[:, ~df.T.duplicated()]
# DRop constant columns
df = df.loc[:, (df != df.iloc[0]).any()]

df = df.sample(frac=1, random_state=42).sort_values(by="Strain").reset_index(drop=True)

Loaded data shape: (571, 1302)


## Data Checks

In [3]:
df.groupby("Strain")["Species"].first().value_counts()

/tmp/ipykernel_65639/512819356.py:1: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("Strain")["Species"].first().value_counts()


Species
Escherichia coli              20
Haemophilus influenzae        18
Enterobacter cloacae          16
Bacillus cereus               10
Yersinia frederiksenii        10
Yersinia enterocolitica       10
Citrobacter freundii          10
Enterobacter asburiae         10
Streptococcus mitis           10
Shigella sonnei               10
Shigella flexneri             10
Listeria monocytogenes        10
Shigella boydii                9
Citrobacter braakii            9
Clostridium glycolicum         9
Haemophilus parainfluenzae     9
Streptococcus oralis           9
Listeria ivanovii              9
Bacillus thuringiensis         8
Clostridium difficile          7
Name: count, dtype: int64

In [4]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 571
Columns: 1084
Use sampling: False (sample size: 571)
Get row duplicates (staged, merged)...
Using top-10 columns for initial filtering: ['peak_list_spectra_bin_838', 'peak_list_spectra_bin_719', 'peak_list_spectra_bin_1261', 'peak_list_spectra_bin_1002', 'Strain', 'peak_list_spectra_bin_695', 'peak_list_spectra_bin_1239', 'peak_list_spectra_bin_430', 'peak_list_spectra_bin_744', 'peak_list_spectra_bin_1163']
Rows remaining as candidates after top-10 filter: 43 (of 571)

#### Duplicate Report
Total duplicate rows: 0 (0.00% of dataset)
Duplicate rows ignoring target: 0 (0.00% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [5]:
# Sample Rows
df_head

,Species,Strain,peak_list_spectra_bin_3,peak_list_spectra_bin_4,peak_list_spectra_bin_5,peak_list_spectra_bin_7,peak_list_spectra_bin_8,peak_list_spectra_bin_9,peak_list_spectra_bin_10,peak_list_spectra_bin_11,peak_list_spectra_bin_12,peak_list_spectra_bin_13,peak_list_spectra_bin_14,peak_list_spectra_bin_15,peak_list_spectra_bin_16,peak_list_spectra_bin_17,peak_list_spectra_bin_18,peak_list_spectra_bin_19,peak_list_spectra_bin_20,peak_list_spectra_bin_21,peak_list_spectra_bin_22,peak_list_spectra_bin_23,peak_list_spectra_bin_24,peak_list_spectra_bin_25,peak_list_spectra_bin_26,peak_list_spectra_bin_28,peak_list_spectra_bin_29,peak_list_spectra_bin_30,peak_list_spectra_bin_31,peak_list_spectra_bin_32,peak_list_spectra_bin_34,peak_list_spectra_bin_35,peak_list_spectra_bin_36,peak_list_spectra_bin_37,peak_list_spectra_bin_39,peak_list_spectra_bin_40,peak_list_spectra_bin_41,peak_list_spectra_bin_42,peak_list_spectra_bin_43,peak_list_spectra_bin_44,peak_list_spectra_bin_45,peak_list_spectra_bin_46,peak_list_spectra_bin_47,peak_list_spectra_bin_48,peak_list_spectra_bin_49,peak_list_spectra_bin_50,peak_list_spectra_bin_52,peak_list_spectra_bin_53,peak_list_spectra_bin_55,peak_list_spectra_bin_56,peak_list_spectra_bin_57,peak_list_spectra_bin_58,peak_list_spectra_bin_59,peak_list_spectra_bin_60,peak_list_spectra_bin_61,peak_list_spectra_bin_62,peak_list_spectra_bin_64,peak_list_spectra_bin_66,peak_list_spectra_bin_67,peak_list_spectra_bin_68,peak_list_spectra_bin_69,peak_list_spectra_bin_70,peak_list_spectra_bin_72,peak_list_spectra_bin_73,peak_list_spectra_bin_74,peak_list_spectra_bin_76,peak_list_spectra_bin_77,peak_list_spectra_bin_78,peak_list_spectra_bin_79,peak_list_spectra_bin_80,peak_list_spectra_bin_81,peak_list_spectra_bin_83,peak_list_spectra_bin_85,peak_list_spectra_bin_86,peak_list_spectra_bin_87,peak_list_spectra_bin_88,peak_list_spectra_bin_89,peak_list_spectra_bin_90,peak_list_spectra_bin_91,peak_list_spectra_bin_92,peak_list_spectra_bin_93,peak_list_spectra_bin_95,peak_list_spectra_bin_97,peak_list_spectra_bin_98,peak_list_spectra_bin_99,peak_list_spectra_bin_100,peak_list_spectra_bin_101,peak_list_spectra_bin_103,peak_list_spectra_bin_105,peak_list_spectra_bin_106,peak_list_spectra_bin_107,peak_list_spectra_bin_108,peak_list_spectra_bin_109,peak_list_spectra_bin_111,peak_list_spectra_bin_112,peak_list_spectra_bin_113,peak_list_spectra_bin_115,peak_list_spectra_bin_116,peak_list_spectra_bin_117,peak_list_spectra_bin_118,peak_list_spectra_bin_119,peak_list_spectra_bin_120,peak_list_spectra_bin_121,peak_list_spectra_bin_122,peak_list_spectra_bin_123,peak_list_spectra_bin_124,peak_list_spectra_bin_125,peak_list_spectra_bin_127,peak_list_spectra_bin_128,peak_list_spectra_bin_130,peak_list_spectra_bin_131,peak_list_spectra_bin_132,peak_list_spectra_bin_133,peak_list_spectra_bin_134,peak_list_spectra_bin_135,peak_list_spectra_bin_137,peak_list_spectra_bin_138,peak_list_spectra_bin_139,peak_list_spectra_bin_140,peak_list_spectra_bin_141,peak_list_spectra_bin_143,peak_list_spectra_bin_145,peak_list_spectra_bin_147,peak_list_spectra_bin_149,peak_list_spectra_bin_150,peak_list_spectra_bin_151,peak_list_spectra_bin_152,peak_list_spectra_bin_153,peak_list_spectra_bin_154,peak_list_spectra_bin_155,peak_list_spectra_bin_156,peak_list_spectra_bin_157,peak_list_spectra_bin_158,peak_list_spectra_bin_159,peak_list_spectra_bin_161,peak_list_spectra_bin_162,peak_list_spectra_bin_163,peak_list_spectra_bin_164,peak_list_spectra_bin_166,peak_list_spectra_bin_167,peak_list_spectra_bin_168,peak_list_spectra_bin_169,peak_list_spectra_bin_170,peak_list_spectra_bin_171,peak_list_spectra_bin_172,peak_list_spectra_bin_173,peak_list_spectra_bin_174,peak_list_spectra_bin_175,peak_list_spectra_bin_176,peak_list_spectra_bin_177,peak_list_spectra_bin_178,peak_list_spectra_bin_180,peak_list_spectra_bin_184,peak_list_spectra_bin_185,peak_list_spectra_bin_186,peak_list_spectra_bin_188,peak_list_spectra_bin_189,peak_list_spectra_bin_190,peak_list_spectra

In [6]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,Species,category,0.0,0.0,20.0,"Escherichia coli, Enterobacter cloacae, Haemophilus influenzae, Shigella flexneri, Listeria monocytogenes, Shigella sonnei, Listeria ivanovii, Enterobacter asburiae, Citrobacter freundii, Yersinia enterocolitica"
1,Strain,category,0.0,0.0,213.0,"7, 66, 126, 188, 49, 211, 198, 59, 56, 18"
2,peak_list_spectra_bin_3,float64,0.0,0.0,87.0,"0.0, 133483.9219, 184903.0469, 1419734.5, 27943.2695, 169226.375, 9010.8008, 11229.2373, 4190.5786, 10624.4219"
3,peak_list_spectra_bin_4,float64,0.0,0.0,8.0,"0.0, 3284.9226, 2330278.25, 72315.8906, 6252400.0, 21472.7812, 58306.8242, 15470.124"
4,peak_list_spectra_bin_5,float64,0.0,0.0,5.0,"0.0, 495329.375, 1260594.75, 312997.5, 825143.0625"
5,peak_list_spectra_bin_7,float64,0.0,0.0,93.0,"0.0, 44431.9219, 4701.7124, 96809.9453, 97883.0859, 205319.7656, 25435.4902, 22785.2656, 34239.082, 38974.2656"
6,peak_list_spectra_bin_8,float64,0.0,0.0,45.0,"0.0, 31592.1621, 4363.2222, 5103.458, 14443.8369, 118729.9453, 506043.9062, 147971.1406, 5474.0811, 55808.3203"
7,peak_list_spectra_bin_9,float64,0.0,0.0,52.0,"0.0, 36870.0352, 408579.4375, 413691.4062, 130932.1641, 29301.8965, 4265.8784, 519936.4375, 67924.7656, 36723.4258"
8,peak_list_spectra_bin_10,float64,0.0,0.0,25.0,"0.0, 67041.6016, 19550.2383, 219985.3906, 56969.9492, 259344.4375, 16830.2324, 32809.9336, 75805.3828, 7972.9575"
9,peak_list_spectra_bin_11,float64,0.0,0.0,55.0,"0.0, 299438.5, 651770.625, 1003640.75, 14568.3535, 125789.6328, 713392.375, 482951.2188, 695641.25, 44889.8516"


In [7]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
peak_list_spectra_bin_3,571.0,7.139591e+04,4.502179e+05,0.0,7.028834e+06
peak_list_spectra_bin_4,571.0,1.533017e+04,2.790895e+05,0.0,6.252400e+06
peak_list_spectra_bin_5,571.0,5.068415e+03,6.752011e+04,0.0,1.260595e+06
peak_list_spectra_bin_7,571.0,2.819248e+04,1.462817e+05,0.0,2.160791e+06
peak_list_spectra_bin_8,571.0,1.016611e+04,8.954112e+04,0.0,1.961872e+06
peak_list_spectra_bin_9,571.0,8.026213e+04,5.129584e+05,0.0,5.417460e+06
peak_list_spectra_bin_10,571.0,8.890188e+03,6.750842e+04,0.0,9.331868e+05
peak_list_spectra_bin_11,571.0,5.524693e+04,2.713508e+05,0.0,3.492954e+06
peak_list_spectra_bin_12,571.0,1.413608e+03,1.500842e+04,0.0,2.705538e+05
peak_list_spectra_bin_13,571.0,4.031135e+04,1.294317e+05,0.0,1.002091e+06


In [8]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column  rank                                      
Species 1           Escherichia coli     60  10.51
        2       Enterobacter cloacae     52   9.11
        3     Haemophilus influenzae     50   8.76
        4          Shigella flexneri     32   5.60
        5     Listeria monocytogenes     31   5.43
Strain  1                          7      6   1.05
        2                         66      6   1.05
        3                        126      6   1.05
        4                        188      6   1.05
        5                         49      5   0.88

In [9]:
# Target Distribution
target_df

,count,pct
Species,,
Escherichia coli,60,10.51
Enterobacter cloacae,52,9.11
Haemophilus influenzae,50,8.76
Shigella flexneri,32,5.60
Listeria monocytogenes,31,5.43
Shigella sonnei,31,5.43
Listeria ivanovii,29,5.08
Enterobacter asburiae,29,5.08
Citrobacter freundii,28,4.90


## Task Curation

In [10]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(
    dataset=df,
    group_on=task_mold.group_on,
    time_on=task_mold.time_on,
    group_labels=task_mold.group_labels,
)
print(f"Recommended splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Providing recommendations based on number of groups (213).
Recommended splits: n_repeats=20, n_splits=3, test_size=None


In [11]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry import curation_recommendations

splits = curation_recommendations.get_recommended_grouped_splits(
    dataset=df,
    n_repeats=n_repeats,
    n_splits=n_splits,
    group_on=task_mold.group_on,
    test_size=none_or_test_size,
    stratify_on=task_mold.stratify_on,
    group_labels=task_mold.group_labels,
    show_splits=True,
    target_on=task_mold.target_column_name,
)

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for grouped data.",
    splits=splits,
)

Using Stratified Grouped splits.
Using label-per-group grouped splits.
Creating index-based splits for 213 groups
Using Stratified IID splits.
Repeat 0, Fold 0:
            Train N: 380, Test N: 191
            Target Distribution:
            	Train target distribution: {'Escherichia coli': 0.11052631578947368, 'Enterobacter cloacae': 0.09736842105263158, 'Haemophilus influenzae': 0.07894736842105263, 'Shigella sonnei': 0.05526315789473684, 'Listeria monocytogenes': 0.05263157894736842, 'Yersinia enterocolitica': 0.05263157894736842, 'Enterobacter asburiae': 0.05263157894736842, 'Citrobacter freundii': 0.05, 'Shigella flexneri': 0.05, 'Listeria ivanovii': 0.05, 'Citrobacter braakii': 0.05, 'Streptococcus mitis': 0.04473684210526316, 'Bacillus cereus': 0.042105263157894736, 'Yersinia frederiksenii': 0.03684210526315789, 'Haemophilus parainfluenzae': 0.03684210526315789, 'Streptococcus oralis': 0.03684210526315789, 'Shigella boydii': 0.034210526315789476, 'Clostridium glycolicum': 0.026

## Export

In [12]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to micro_mass/019d319c-d4d7-7b90-9c84-1201e1f4eb51
019d319c-d4d7-7b90-9c84-1201e1f4eb51
a869e9ec6842f4666caa1774b744370a6979c9f5e8595895607a79afb23bd0fd
